<a href="https://colab.research.google.com/github/attabeezy/crop-guard/blob/main/notebooks/06_export_float32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CropGuard — float32 TFLite export and parity

The full-INT8 artifact failed its deployment parity gate. This notebook exports the locked fine-tuned checkpoint to float32 TensorFlow Lite, evaluates it over all 4,811 held-out test images, and packages an Android tensor contract.

The evaluated temperature and confidence threshold remain unchanged. No model or policy tuning occurs here.

## 1. Setup

A CPU runtime is sufficient.

In [ ]:
%pip install -q kagglehub pandas scikit-learn pillow tqdm

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 2026
IMAGE_SIZE = 224
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)

## 2. Download and verify immutable inputs

In [ ]:
import hashlib
import json
import shutil
import urllib.request
import zipfile

AUDIT_COMMIT = "4c8b366df4c251d6424258c3cb74a58f2289ee16"
AUDIT_SHA256 = "1f69013b0a0fc58cba5dccab37d50e8767902580f443ac13daa3fba885e3269c"
MODEL_COMMIT = "08d86ed7807075b195a43acb80e11ef738ee4b28"
MODEL_SHA256 = "14c7ed601c3bebd64238a1a3375c1653bc6017009df645a22878241525e0aa9d"
EVALUATION_COMMIT = "519f759b7fad88668f9ec5511dc631ccf3e81494"
EVALUATION_SUMMARY_SHA256 = "b7c9c9ac4607d1b59f606787f20e89550bcbd1fdb57c2cf9d17fee97e967345d"
TEST_PREDICTIONS_SHA256 = "c5ed095091f5574f261286f7a7d6f7687061233ecdd7320e3b567f8a7ba6e57a"

input_dir = Path("/content/cropguard_float_export_inputs")
if input_dir.exists():
    shutil.rmtree(input_dir)
input_dir.mkdir()

def download(url, destination, expected_sha256):
    urllib.request.urlretrieve(url, destination)
    actual = hashlib.sha256(Path(destination).read_bytes()).hexdigest()
    if actual != expected_sha256:
        raise RuntimeError(f"Hash mismatch for {destination}: {actual}")

audit_zip = input_dir / "audit.zip"
download(
    f"https://raw.githubusercontent.com/attabeezy/crop-guard/{AUDIT_COMMIT}/cropguard-data-prep.zip",
    audit_zip, AUDIT_SHA256,
)
audit_dir = input_dir / "audit"
with zipfile.ZipFile(audit_zip) as archive:
    archive.extractall(audit_dir)

keras_path = input_dir / "best_finetuned.keras"
download(
    "https://media.githubusercontent.com/media/attabeezy/crop-guard/"
    f"{MODEL_COMMIT}/results/training-artifacts/best_finetuned.keras",
    keras_path, MODEL_SHA256,
)

summary_path = input_dir / "evaluation_summary.json"
predictions_path = input_dir / "test_predictions.csv"
download(
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{EVALUATION_COMMIT}/results/evaluation/evaluation_summary.json",
    summary_path, EVALUATION_SUMMARY_SHA256,
)
download(
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{EVALUATION_COMMIT}/results/evaluation/test_predictions.csv",
    predictions_path, TEST_PREDICTIONS_SHA256,
)
evaluation = json.loads(summary_path.read_text())
test_predictions = pd.read_csv(predictions_path)
temperature = float(evaluation["calibration"]["temperature"])
threshold = float(evaluation["test"]["confidence_threshold_locked_on_validation"])
classes = sorted(test_predictions["label"].unique())
class_to_index = {label: index for index, label in enumerate(classes)}
assert len(classes) == 22
print("All pinned inputs verified")

## 3. Resolve and verify the evaluated test images

In [ ]:
import kagglehub
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

DATASET_HANDLE = "nirmalsankalana/crop-pest-and-disease-detection"
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
manifest = pd.read_csv(audit_dir / "test.csv")
test_frame = test_predictions[["path"]].merge(
    manifest, on="path", how="left", validate="one_to_one"
)
test_frame["absolute_path"] = test_frame["path"].map(lambda value: str(dataset_root / value))
test_frame["target"] = test_frame["label"].map(class_to_index)
if test_frame["label"].isna().any() or test_frame["target"].isna().any():
    raise RuntimeError("Test predictions do not match the audit manifest")

def verify_image(row):
    path = Path(row.absolute_path)
    if not path.is_file():
        return f"missing: {row.path}"
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    return None if actual == row.sha256 else f"hash mismatch: {row.path}"

rows = list(test_frame.itertuples(index=False))
with ThreadPoolExecutor(max_workers=8) as pool:
    problems = list(tqdm(pool.map(verify_image, rows), total=len(rows)))
problems = [problem for problem in problems if problem]
if problems:
    raise RuntimeError(problems[:5])
print(f"Verified {len(test_frame):,} held-out images")

## 4. Convert the selected checkpoint to float32 TFLite

In [ ]:
model = tf.keras.models.load_model(keras_path, compile=False)
if model.input_shape != (None, IMAGE_SIZE, IMAGE_SIZE, 3):
    raise RuntimeError(f"Unexpected input shape: {model.input_shape}")
if model.output_shape[-1] != len(classes):
    raise RuntimeError(f"Unexpected output shape: {model.output_shape}")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
export_dir = Path("/content/cropguard_float32_export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir()
tflite_path = export_dir / "model.tflite"
tflite_path.write_bytes(tflite_bytes)
print(f"Float32 TFLite size: {len(tflite_bytes) / 1_000_000:.2f} MB")
print("SHA-256:", hashlib.sha256(tflite_bytes).hexdigest())

## 5. Enforce the tensor contract

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_path), num_threads=4)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
if input_detail["dtype"] != np.float32 or output_detail["dtype"] != np.float32:
    raise RuntimeError(
        f"Expected float32 tensors; got {input_detail['dtype']} and {output_detail['dtype']}"
    )
if tuple(input_detail["shape"]) != (1, IMAGE_SIZE, IMAGE_SIZE, 3):
    raise RuntimeError(f"Unexpected input tensor: {input_detail['shape']}")
if tuple(output_detail["shape"]) != (1, len(classes)):
    raise RuntimeError(f"Unexpected output tensor: {output_detail['shape']}")
operator_names = [item["op_name"] for item in interpreter._get_ops_details()]
flex_ops = [name for name in operator_names if name.startswith("Flex")]
if flex_ops:
    raise RuntimeError(f"Model requires unsupported TensorFlow Flex ops: {sorted(set(flex_ops))}")
print("Input:", input_detail["shape"], input_detail["dtype"])
print("Output:", output_detail["shape"], output_detail["dtype"])
print("TFLite built-in operators:", sorted(set(operator_names)))

## 6. Full held-out parity evaluation

In [ ]:
def load_float_image(path):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
    return tf.cast(image, tf.float32).numpy()[None, ...]

def run_tflite(path):
    interpreter.set_tensor(input_detail["index"], load_float_image(path))
    interpreter.invoke()
    return interpreter.get_tensor(output_detail["index"])[0]

tflite_probabilities = np.stack([
    run_tflite(path) for path in tqdm(test_frame["absolute_path"], desc="Float32 TFLite")
])
tflite_predictions = tflite_probabilities.argmax(axis=1)
truth = test_frame["target"].to_numpy(dtype=np.int32)
reference_predictions = test_predictions["predicted_label"].map(class_to_index).to_numpy()
print("Completed parity inference:", tflite_probabilities.shape)

## 7. Apply the locked confidence policy and release gates

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def temperature_scale(probabilities, value):
    logits = np.log(np.clip(probabilities, 1e-8, 1.0)) / value
    logits -= logits.max(axis=1, keepdims=True)
    exponentiated = np.exp(logits)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)

calibrated = temperature_scale(tflite_probabilities, temperature)
confidence = calibrated.max(axis=1)
accepted = confidence >= threshold
correct = tflite_predictions == truth
reference_accuracy = float(evaluation["test"]["accuracy"])
reference_macro_f1 = float(evaluation["test"]["macro_f1"])
tflite_accuracy = float(accuracy_score(truth, tflite_predictions))
tflite_macro_f1 = float(f1_score(truth, tflite_predictions, average="macro"))
reference_confidence = test_predictions["raw_confidence"].to_numpy(dtype=float)

parity = {
    "reference_accuracy": reference_accuracy,
    "tflite_accuracy": tflite_accuracy,
    "accuracy_drop": reference_accuracy - tflite_accuracy,
    "reference_macro_f1": reference_macro_f1,
    "tflite_macro_f1": tflite_macro_f1,
    "macro_f1_drop": reference_macro_f1 - tflite_macro_f1,
    "prediction_agreement": float((tflite_predictions == reference_predictions).mean()),
    "changed_predictions": int((tflite_predictions != reference_predictions).sum()),
    "mean_absolute_raw_confidence_delta": float(
        np.mean(np.abs(tflite_probabilities.max(axis=1) - reference_confidence))
    ),
    "coverage": float(accepted.mean()),
    "accepted_accuracy": float(correct[accepted].mean()) if accepted.any() else None,
}
LIMITS = {
    "maximum_accuracy_drop": 0.001,
    "maximum_macro_f1_drop": 0.001,
    "minimum_prediction_agreement": 0.999,
    "maximum_mean_confidence_delta": 0.001,
}
release_eligible = (
    parity["accuracy_drop"] <= LIMITS["maximum_accuracy_drop"]
    and parity["macro_f1_drop"] <= LIMITS["maximum_macro_f1_drop"]
    and parity["prediction_agreement"] >= LIMITS["minimum_prediction_agreement"]
    and parity["mean_absolute_raw_confidence_delta"] <= LIMITS["maximum_mean_confidence_delta"]
)
parity["status"] = "pass" if release_eligible else "fail"
display(parity)
if not release_eligible:
    print("RELEASE GATE FAILED: do not integrate this artifact.")

## 8. Host-runtime benchmark

This measures Colab CPU inference only; it is not an Android latency claim.

In [ ]:
import time

benchmark_input = load_float_image(test_frame.iloc[0]["absolute_path"])
for _ in range(10):
    interpreter.set_tensor(input_detail["index"], benchmark_input)
    interpreter.invoke()
latencies = []
for _ in range(100):
    interpreter.set_tensor(input_detail["index"], benchmark_input)
    started = time.perf_counter()
    interpreter.invoke()
    latencies.append((time.perf_counter() - started) * 1000)
latency = {
    "environment": "Google Colab CPU; not Android device latency",
    "threads": 4,
    "runs": 100,
    "median_ms": float(np.median(latencies)),
    "p95_ms": float(np.percentile(latencies, 95)),
}
display(latency)

## 9. Package Android assets

In [ ]:
DISPLAY_OVERRIDES = {
    "healthy": "Healthy",
    "fall_armyworm": "Fall armyworm damage",
    "grasshopper": "Grasshopper damage",
    "leaf_beetle": "Leaf beetle damage",
    "green_mite": "Green mite damage",
    "leaf_miner": "Leaf miner damage",
}
with (export_dir / "labels.txt").open("w", encoding="utf-8") as stream:
    for label in classes:
        crop, key = label.split("|", 1)
        display = DISPLAY_OVERRIDES.get(key, key.replace("_", " ").title())
        stream.write(f"{crop}|{key}|{crop.title()} — {display}\n")

contract = {
    "model_sha256": hashlib.sha256(tflite_bytes).hexdigest(),
    "model_size_bytes": len(tflite_bytes),
    "input": {
        "shape": input_detail["shape"].tolist(),
        "dtype": "float32",
        "source_pixel_range": [0.0, 255.0],
        "resize": [IMAGE_SIZE, IMAGE_SIZE],
    },
    "output": {
        "shape": output_detail["shape"].tolist(),
        "dtype": "float32",
        "class_order_file": "labels.txt",
    },
    "postprocessing": {
        "temperature": temperature,
        "confidence_threshold": threshold,
        "formula": "softmax(log(clip(probabilities,1e-8,1))/temperature)",
    },
}
(export_dir / "tensor_contract.json").write_text(json.dumps(contract, indent=2))

report = {
    "tensorflow_version": tf.__version__,
    "source_model_commit": MODEL_COMMIT,
    "source_model_sha256": MODEL_SHA256,
    "audit_commit": AUDIT_COMMIT,
    "evaluation_commit": EVALUATION_COMMIT,
    "format": "TensorFlow Lite float32",
    "parity_limits": LIMITS,
    "parity": parity,
    "latency": latency,
    "release_eligible": release_eligible,
    "int8_status": "rejected after failed deployment parity",
}
(export_dir / "deployment_report.json").write_text(json.dumps(report, indent=2))

parity_predictions = test_predictions[["path", "label", "predicted_label"]].copy()
parity_predictions["tflite_predicted_label"] = [classes[index] for index in tflite_predictions]
parity_predictions["tflite_confidence"] = confidence
parity_predictions["accepted"] = accepted
parity_predictions["matches_reference"] = tflite_predictions == reference_predictions
parity_predictions.to_csv(export_dir / "float32_parity_predictions.csv", index=False)

archive = shutil.make_archive("/content/cropguard-float32-export", "zip", export_dir)
print(f"Created {archive} ({Path(archive).stat().st_size / 1_000_000:.1f} MB)")
print("Archive SHA-256:", hashlib.sha256(Path(archive).read_bytes()).hexdigest())

In [ ]:
from google.colab import files
files.download("/content/cropguard-float32-export.zip")

## Next step

Provide the downloaded ZIP for inspection. Integrate it into Android only when `deployment_report.json` reports `release_eligible: true`.